In [1]:
#Imports 
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import find_peaks
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from scipy.optimize import minimize
from scipy.stats import norm 
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
import random
from random import sample
import wandb


In [2]:
# Load data
import pickle
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import f1_score
from sklearn import preprocessing
import numpy as np
import matplotlib.pyplot as plt
with open("data (1).pkl", "rb") as f:
    data = pickle.load(f)

x = np.array(data['x'])
y = np.array(data['y'])
y = y[:x.shape[0]]  # Trim y to match x if necessary

# Subsets (optional, not used directly in model training)
ai_data = x[y == 0]
human_data = x[y == 1]

train_data=np.loadtxt("ECG200_TRAIN.txt")
test_data=np.loadtxt("ECG200_TEST.txt")

In [ ]:
#MLP ECG Data

"""
Bayesian Optimisation of interval centres for time‑series classification.

Given multivariate patient time‑series (rows = patients, columns = time),
we search for the best fixed‑width interval that yields the lowest test error
for a simple neural network classifier. We use a Gaussian Process (GP) surrogate
with a Matérn(ν=1.5) kernel and the Expected Improvement (EI) acquisition.

Key steps:
1) Sample a small set of interval centres (force edges + a few random).
2) For each centre, train an MLP on the corresponding interval and record error.
3) Fit a GP to map centre → error.
4) Use EI to propose the next centre to evaluate, avoiding duplicates.
5) Stop automatically when a regret‑style bound is below the empirical variance.
"""

from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from scipy.stats import norm

# ---------------------------------------------------------------------
# 0) Inputs expected:
#    - train_data: NumPy array with shape (n_samples, 1 + T)
#      Column 0 = labels (classification targets)
#      Columns 1..T = time‑series features for each patient
# ---------------------------------------------------------------------

# Extract labels and patient data from the provided array
labels: np.ndarray = train_data[:, 0]
patient_data: np.ndarray = train_data[:, 1:]

# Sanity checks to make debugging easier for future readers
assert patient_data.ndim == 2, "patient_data must be 2D: [n_patients, time]"
assert labels.shape[0] == patient_data.shape[0], "Labels must align with rows in patient_data"

# ---------------------------------------------------------------------
# 1) Interval configuration and initial sampling
# ---------------------------------------------------------------------

# Fixed interval width (number of time steps used to train the classifier)
interval_width: int = 10  # keep even for symmetric slicing

# Always include the two ends of the domain to anchor the GP at the boundaries
edge_points = [0, patient_data.shape[1] - 1]

# Choose a handful of random interior centres (without replacement)
remaining_points = np.setdiff1d(np.arange(patient_data.shape[1]), edge_points)
random_points = np.random.choice(remaining_points, size=8, replace=False)

# Initial design of experiment: edges + random interior points
sampled_centers = np.concatenate([edge_points, random_points])

# ---------------------------------------------------------------------
# 2) Helper: map a centre to a [start, end) interval
# ---------------------------------------------------------------------
def get_sampled_interval(patient_data: np.ndarray, center: int, interval_width: int) -> tuple[int, int]:
    """
    Given a centre index and a desired width, return a safe [start, end) slice
    within the time dimension. Uses half‑width on each side and clips to bounds.

    Note: with an even width, the 'centre' is the left of the two middle points.
    Near the edges we accept narrower windows due to clipping.
    """
    half = interval_width // 2
    start = max(0, center - half)
    end = min(patient_data.shape[1], center + half)
    # If you need *exactly* interval_width, you could re‑shift start/end here.
    return start, end

# ---------------------------------------------------------------------
# 3) Objective: train a small MLP on the interval and return the error rate
# ---------------------------------------------------------------------
def train_nn(patient_data: np.ndarray, labels: np.ndarray, center: int) -> float:
    """
    Train an MLP on the sub‑interval defined by 'center' and report test error.

    Splits patients into train/test with stratification for balanced classes.
    Uses accuracy on the held‑out set, then returns error = 1 − accuracy.
    """
    start, end = get_sampled_interval(patient_data, center, interval_width)

    # Slice the time interval for all patients
    X = patient_data[:, start:end]
    y = labels

    # Defensive check: ensure the interval contains at least one column
    if X.shape[1] < 1:
        return 1.0  # degenerate interval, treat as maximally bad

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # A compact MLP; increase capacity or tune if underfitting
    nn_model = MLPClassifier(hidden_layer_sizes=(50,), max_iter=1000, random_state=42)
    nn_model.fit(X_train, y_train)

    y_pred = nn_model.predict(X_test)
    error_rate = 1.0 - accuracy_score(y_test, y_pred)
    return float(error_rate)

# ---------------------------------------------------------------------
# 4) Evaluate the initial design
# ---------------------------------------------------------------------
nn_results = [
    {"Interval center": int(center), "Error Rate": train_nn(patient_data, labels, int(center))}
    for center in sampled_centers
]
nn_df = pd.DataFrame(nn_results).sort_values(by="Interval center").reset_index(drop=True)

# Regression data for the GP surrogate
# sklearn's GPR prefers y to be 1D shape (n,), so we ravel for cleanliness.
x_sample = nn_df["Interval center"].to_numpy(dtype=float).reshape(-1, 1)
y_sample = nn_df["Error Rate"].to_numpy(dtype=float)  # shape (n,)

# GP with Matérn kernel; alpha is a small nugget to stabilise inversion
kernel = Matern(nu=1.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, n_restarts_optimizer=10, random_state=42)

# ---------------------------------------------------------------------
# 5) Acquisition: Expected Improvement for *minimisation* of error
#    IMPORTANT FIX: EI must use the *current best (minimum) error*.
# ---------------------------------------------------------------------
def expected_improvement(
    X: np.ndarray,
    x_sample: np.ndarray,
    y_sample: np.ndarray,
    gp: GaussianProcessRegressor,
    xi: float = 0.01,
) -> np.ndarray:
    """
    Compute EI at candidate points X for *minimising* the objective.
    EI(x) = E[max(0, f_best − f(x) − xi)], using GP posterior mean/variance.

    xi encourages exploration; larger xi explores more.
    """
    mu, sigma = gp.predict(X, return_std=True)
    f_best = np.min(y_sample)  # best observed error so far (lower is better)

    # Avoid division by zero; vectorised safe computation
    with np.errstate(divide="ignore"):
        z = (f_best - mu - xi) / sigma
        ei = (f_best - mu - xi) * norm.cdf(z) + sigma * norm.pdf(z)
        ei[sigma == 0.0] = 0.0
    return ei

# ---------------------------------------------------------------------
# 6) Propose the next centre on a discrete grid, avoiding duplicates
# ---------------------------------------------------------------------
def bayesian_sample_one_point(
    x_sample: np.ndarray,
    y_sample: np.ndarray,
    gp: GaussianProcessRegressor,
    bounds: tuple[int, int],
    tried_points: set[int],
) -> int | None:
    """
    Evaluate EI on the integer grid between bounds and return the argmax
    that has not been tried yet. Returns None if all points are exhausted.
    """
    x_candidates = np.arange(int(bounds[0]), int(bounds[1]) + 1).reshape(-1, 1)
    ei_values = expected_improvement(x_candidates, x_sample, y_sample, gp).ravel()

    # Mask out centres we have already evaluated
    mask = np.isin(x_candidates.ravel(), list(tried_points))
    ei_values_masked = np.where(mask, -np.inf, ei_values)

    if np.all(ei_values_masked == -np.inf):
        print("All possible points have been sampled. Ending optimisation.")
        return None

    best_idx = int(np.argmax(ei_values_masked))
    return int(x_candidates[best_idx, 0])

# Simple, observable variance proxy from the realised error rates
def estimate_variance(error_rates: pd.Series | np.ndarray) -> float:
    """Unbiased sample variance can be used as well; this is a basic proxy."""
    arr = np.asarray(error_rates, dtype=float)
    return float(np.var(arr))

# ---------------------------------------------------------------------
# 7) Main BO loop with automatic termination
# ---------------------------------------------------------------------
iteration = 0
max_iterations = 100

while iteration < max_iterations:
    iteration += 1

    # Fit the surrogate on all observations so far
    gp.fit(x_sample, y_sample)

    # Propose a new centre using EI over the integer grid
    bounds = (0, patient_data.shape[1] - 1)
    tried_points = set(nn_df["Interval center"].astype(int).tolist())
    new_center = bayesian_sample_one_point(x_sample, y_sample, gp, bounds, tried_points)
    if new_center is None:
        break  # All grid points exhausted

    # Evaluate the true objective at the proposed centre
    new_error_rate = train_nn(patient_data, labels, new_center)

    # Add to our dataset, dropping accidental duplicates
    nn_df = pd.concat(
        [nn_df, pd.DataFrame([{"Interval center": new_center, "Error Rate": new_error_rate}])]
    ).drop_duplicates(subset=["Interval center"], keep="first").sort_values("Interval center").reset_index(drop=True)

    # Update training arrays for the GP
    x_sample = nn_df["Interval center"].to_numpy(dtype=float).reshape(-1, 1)
    y_sample = nn_df["Error Rate"].to_numpy(dtype=float)

    # Compute a simple regret‑style bound using GP posterior over a dense grid
    X_plot = np.linspace(bounds[0], bounds[1], 200).reshape(-1, 1)
    mu, sigma = gp.predict(X_plot, return_std=True)

    # Heuristic bound: best UCB minus best LCB over the domain
    lcb = mu - 1.0 * sigma
    ucb = mu + 1.0 * sigma
    simple_regret_bound = float(np.min(ucb) - np.min(lcb))

    # Empirical variance of observed errors
    error_var = estimate_variance(nn_df["Error Rate"])

    # Progress logging for transparency
    print(f"Iteration {iteration}: New Interval Center = {new_center}, Error Rate = {new_error_rate:.4f}")
    print(f"Simple Regret Bound: {simple_regret_bound:.6f}, Error Variance: {error_var:.6f}")

    # Automatic stopping: if model uncertainty is smaller than noise proxy
    if simple_regret_bound <= np.sqrt(error_var):
        print("Termination Condition Met: Stopping Bayesian Optimisation.")
        break

    # ---------------------------------------------------------------
    # Visualisation: surrogate fit and acquisition
    # ---------------------------------------------------------------
    plt.figure(figsize=(12, 6))
    plt.plot(x_sample, y_sample, "ro", label="Sampled points")
    plt.plot([new_center], [new_error_rate], "go", label="New sampled point")
    plt.plot(X_plot, mu, "b-", label="GP mean prediction")
    plt.fill_between(
        X_plot.ravel(),
        mu - 1.96 * sigma,
        mu + 1.96 * sigma,
        alpha=0.2,
        label="95% GP confidence"
    )
    plt.xlabel("Interval centre")
    plt.ylabel("Error rate")
    plt.title("Bayesian Optimisation for Interval Centres")
    plt.legend()
    plt.grid(True)
    plt.show()

    ei_values = expected_improvement(X_plot, x_sample, y_sample, gp)
    plt.figure(figsize=(12, 6))
    plt.plot(X_plot, ei_values, "g-", label="Expected Improvement")
    plt.xlabel("Interval centre")
    plt.ylabel("EI")
    plt.title("Acquisition Function")
    plt.legend()
    plt.grid(True)
    plt.show()

# ---------------------------------------------------------------------
# 8) Report the best interval found
# ---------------------------------------------------------------------
best_row = nn_df.loc[nn_df["Error Rate"].idxmin()]
best_center = int(best_row["Interval center"])
best_error = float(best_row["Error Rate"])
best_start, best_end = get_sampled_interval(patient_data, best_center, interval_width)

print("\nBest Interval Found:")
print(f"Centre: {best_center}  →  Interval: [{best_start}, {best_end})")
print(f"Error Rate: {best_error:.4f}")


In [ ]:
#CNN ECG Data

# optimised CNN ECG Data
"""
End-to-end pipeline for finding the best time-window (interval centre) on ECG200
for a 1D CNN classifier. We:
  1) Load ECG200 train/test and combine for a single pool (as in the original code).
  2) Define a compact 1D CNN (two Conv/Pool blocks → MLP head).
  3) Define an objective: train the CNN on a fixed-width interval and return error.
  4) Run Bayesian Optimisation (Matérn GP + Expected Improvement) over centres.
  5) Report the best interval against a full-window baseline.

Notes for readers:
- Labels in ECG200 are {-1, +1}. We remap to {0, 1} for BCEWithLogitsLoss.
- Standardisation is done *after* the train/test split to avoid leakage.  [FIX]
- Stratification requires 1D labels; we use ravel() when splitting.       [FIX]
- BO treats the objective as a minimisation of error (lower is better).
"""

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern


# ---------------------------------------------------------------------
# 1) Load ECG200 and form a single dataset (features X, labels y ∈ {0,1})
# ---------------------------------------------------------------------
train_data = np.loadtxt("ECG200_TRAIN.txt")   # shape: (n_train, 1 + T)
test_data  = np.loadtxt("ECG200_TEST.txt")    # shape: (n_test,  1 + T)

train_labels   = train_data[:, 0]
train_features = train_data[:, 1:]
test_labels    = test_data[:, 0]
test_features  = test_data[:, 1:]

# Stack and convert: X is float32 (PyTorch-friendly), y is {0,1} as int
x = np.vstack((train_features, test_features)).astype(np.float32)
y = ((np.concatenate((train_labels, test_labels)) + 1) / 2).astype(int)  # -1→0, +1→1


# ---------------------------------------------------------------------
# 2) Optimised CNN definition (two Conv1d blocks → FC layers)
# ---------------------------------------------------------------------
class CNN1D(nn.Module):
    """
    A light 1D CNN for sequence classification:
      Input: (batch, 1, L)
      Stem:  Conv1d(1→laten, k=3, pad=1), ReLU, MaxPool(2)
             Conv1d(laten→laten, k=3, pad=1), ReLU, MaxPool(2)
      Head:  Flatten → Linear → ReLU → Dropout → Linear(→1 logit)

    The final layer outputs a single logit; apply sigmoid at evaluation time.
    """
    def __init__(self, input_length: int, laten: int = 10):
        super().__init__()
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=laten, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool1d(kernel_size=2)
        self.conv2 = nn.Conv1d(in_channels=laten, out_channels=laten, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool1d(kernel_size=2)
        self.flatten = nn.Flatten()

        # Compute flattened size after conv/pool blocks using a dummy pass
        conv_output_length = self._get_conv_output_shape(input_length)
        self.fc1 = nn.Linear(conv_output_length, laten)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(laten, 1)  # 1 logit for binary classification

    def _get_conv_output_shape(self, input_length: int) -> int:
        """Dry-run forward over zeros to determine the flattened dimension."""
        dummy = torch.zeros(1, 1, input_length)           # (batch=1, channels=1, length=L)
        x = self.pool1(torch.relu(self.conv1(dummy)))
        x = self.pool2(torch.relu(self.conv2(x)))
        return x.view(1, -1).size(1)                      # total features after flatten

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward expects x as (batch, L, 1) for convenience.
        We permute to (batch, 1, L) to satisfy Conv1d's (N, C, L) convention.
        """
        x = x.permute(0, 2, 1)                # (N, 1, L)
        x = self.pool1(torch.relu(self.conv1(x)))
        x = self.pool2(torch.relu(self.conv2(x)))
        x = self.flatten(x)
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)                     # raw logits (no sigmoid here)


# ---------------------------------------------------------------------
# 3) Objective: train CNN on a given interval centre/width, return error
# ---------------------------------------------------------------------
def train_nn(data: np.ndarray, labels: np.ndarray, center: int, width: int) -> float:
    """
    Train the CNN on a fixed-width interval around 'center' and report error rate
    on a held-out test split.

    Steps:
    - Slice the interval [start, end) safely.
    - Split patients into train/test with stratification.
    - Standardise based on the *training* split only.                      [FIX]
    - Train for a small, fixed number of epochs.
    - Compute accuracy on test; return error = 1 - accuracy.

    Returns:
        float: error rate in [0,1]. If the interval is too short (<6), returns 1.0.
    """
    # Safe interval bounds
    start = max(0, center - width // 2)
    end   = min(data.shape[1], center + width // 2)

    # Fail-safe: very short windows likely won't train well
    if end - start < 6:
        return 1.0

    # Extract interval features (N, L_interval) and binary labels (N, 1)
    x_interval = data[:, start:end]
    y_binary   = labels.astype(np.float32).reshape(-1, 1)

    # Train/test split with proper 1D labels for stratify                 [FIX]
    X_tr, X_te, y_tr, y_te = train_test_split(
        x_interval, y_binary, test_size=0.2, stratify=y_binary.ravel(), random_state=42
    )

    # Standardise features using the *training* data only to avoid leakage [FIX]
    scaler = StandardScaler().fit(X_tr)
    X_tr = scaler.transform(X_tr)[:, :, np.newaxis]   # → (N_tr, L, 1)
    X_te = scaler.transform(X_te)[:, :, np.newaxis]   # → (N_te, L, 1)

    # Numpy → Torch tensors (CPU by default)
    X_tr = torch.tensor(X_tr, dtype=torch.float32)
    X_te = torch.tensor(X_te, dtype=torch.float32)
    y_tr = torch.tensor(y_tr, dtype=torch.float32)
    y_te = torch.tensor(y_te, dtype=torch.float32)

    # Model, loss (with logits), and optimiser
    model = CNN1D(input_length=X_tr.shape[1])
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    # Mini-batch loader
    train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=32, shuffle=True)

    # Training loop (small, fixed number of epochs)
    model.train()
    for epoch in range(15):
        for xb, yb in train_loader:
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

    # Evaluation: probability → class → accuracy → error
    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(model(X_te)).squeeze(dim=-1)  # (N_te,)
        preds = (probs > 0.5).int().cpu().numpy()
    return 1.0 - accuracy_score(y_te.cpu().numpy(), preds)


# ---------------------------------------------------------------------
# 4) Baseline: train CNN on the full window and report error
# ---------------------------------------------------------------------
def train_full_nn(data: np.ndarray, labels: np.ndarray) -> float:
    """
    Train the same CNN on the full-length signal as a baseline.
    Mirrors train_nn but uses the entire time axis.
    """
    y_binary = labels.astype(np.float32).reshape(-1, 1)

    # Train/test split first, then fit scaler on train only               [FIX]
    X_tr, X_te, y_tr, y_te = train_test_split(
        data, y_binary, test_size=0.2, stratify=y_binary.ravel(), random_state=42
    )

    scaler = StandardScaler().fit(X_tr)
    X_tr = scaler.transform(X_tr)[:, :, np.newaxis]
    X_te = scaler.transform(X_te)[:, :, np.newaxis]

    X_tr = torch.tensor(X_tr, dtype=torch.float32)
    X_te = torch.tensor(X_te, dtype=torch.float32)
    y_tr = torch.tensor(y_tr, dtype=torch.float32)
    y_te = torch.tensor(y_te, dtype=torch.float32)

    model = CNN1D(input_length=X_tr.shape[1])
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=32, shuffle=True)

    model.train()
    for epoch in range(15):
        for xb, yb in train_loader:
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        probs = torch.sigmoid(model(X_te)).squeeze(dim=-1)
        preds = (probs > 0.5).int().cpu().numpy()
    return 1.0 - accuracy_score(y_te.cpu().numpy(), preds)


# ---------------------------------------------------------------------
# 5) Bayesian Optimisation over interval centres (discrete grid)
# ---------------------------------------------------------------------
interval_width = 20                 # fixed window width (in time steps)
n_initial = 10                      # initial random design size

center_bounds = (0, x.shape[1] - 1) # search domain for centres (inclusive)

# Initial centres sampled uniformly from the domain (inclusive of upper bound) [FIX]
initial_centers = np.random.choice(
    np.arange(center_bounds[0], center_bounds[1] + 1), size=n_initial, replace=False
)

# Evaluate the objective at initial centres
x_sample = initial_centers.reshape(-1, 1)                           # (n0, 1)
y_sample = np.array([train_nn(x, y, c, interval_width) for c in initial_centers], dtype=float)  # (n0,)


def expected_improvement(X: np.ndarray,
                         X_sample: np.ndarray,
                         Y_sample: np.ndarray,
                         model: GaussianProcessRegressor,
                         xi: float = 0.01) -> np.ndarray:
    """
    Expected Improvement for *minimisation*.
    EI(x) = E[max(0, f_best − f(x) − xi)], with f the GP posterior.
    """
    mu, sigma = model.predict(X, return_std=True)   # posterior mean/std at candidates
    f_best = np.min(Y_sample)                       # best (lowest) observed error
    with np.errstate(divide='warn'):
        imp = f_best - mu - xi
        Z = imp / sigma
        ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
        ei[sigma == 0.0] = 0.0
    return ei


# BO loop configuration: GP surrogate with Matérn(ν=1.5)
kernel = Matern(nu=1.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, n_restarts_optimizer=10, random_state=42)
max_iter = 30

for _ in range(max_iter):
    # Fit GP to current observations (centres → errors)
    gp.fit(x_sample, y_sample)

    # Candidate grid across all integer centres; remove already tried
    grid = np.arange(center_bounds[0], center_bounds[1] + 1).reshape(-1, 1)
    tried = set(int(c) for c in x_sample.ravel())
    grid = np.array([pt for pt in grid if int(pt[0]) not in tried])
    if len(grid) == 0:
        break  # all centres exhausted

    # Select next point by maximising EI
    ei = expected_improvement(grid, x_sample, y_sample, gp)
    best_idx = int(np.argmax(ei))
    next_center = int(grid[best_idx][0])

    # Evaluate true objective and append to our dataset
    new_error = train_nn(x, y, next_center, interval_width)
    x_sample = np.vstack((x_sample, [[next_center]]))
    y_sample = np.append(y_sample, new_error)


# ---------------------------------------------------------------------
# 6) Best result and baseline comparison
# ---------------------------------------------------------------------
best_idx = int(np.argmin(y_sample))
best_center = int(x_sample[best_idx][0])

best_start = max(0, best_center - interval_width // 2)
best_end   = min(x.shape[1], best_center + interval_width // 2)

best_error = float(y_sample[best_idx])
full_model_error = train_full_nn(x, y)

print(" Best interval center:", best_center)
print(" Best interval range:  ({} to {})".format(best_start, best_end))
print(" Best interval error:  {:.4f}".format(best_error))
print(" Full model error:     {:.4f}".format(full_model_error))


In [ ]:
#MLP AI Data


# === Imports ===
# These libraries cover numerical operations, data handling, plotting, model building,
# Gaussian Process regression for Bayesian Optimisation, and probability calculations.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern
from scipy.stats import norm
import pickle

# === Data preparation ===
# 'data' should be a dictionary-like object with keys "x" (feature matrix) and "y" (labels).
# Features are converted to NumPy arrays for efficiency.
x = np.array(data["x"], dtype=np.float64)
y = np.array(data["y"])

# Remove any rows with NaN values in the feature set to ensure model training doesn't fail.
tokeep = set(range(x.shape[0])).difference(set(np.where(np.isnan(x))[0]))
x = x[list(tokeep)]
y = y[list(tokeep)]

# === Interval sampling parameters ===
# We will train the model only on a sub-window ("interval") of features at a time.
interval_width = 40  # Number of consecutive features to include in each interval.

# Select initial set of "interval centres" for evaluation.
# Always include the first and last feature positions (edge_points),
# then sample a few random internal positions.
edge_points = [0, x.shape[1] - 1]
remaining_points = np.setdiff1d(np.arange(x.shape[1]), edge_points)
random_points = np.random.choice(remaining_points, size=8, replace=False)
sampled_centers = np.concatenate([edge_points, random_points])

# Helper: Given a centre position, compute start and end indices of the interval.
def get_sampled_interval(x, center, interval_width):
    start = max(0, center - interval_width // 2)
    end = min(x.shape[1], center + interval_width // 2)
    return start, end

# === Neural network training function ===
# Trains an MLP on the sub-window of features around a given centre and returns the error rate.
def train_nn(x, y, center):
    start, end = get_sampled_interval(x, center, interval_width)
    x_interval = x[:, start:end]

    # Stratified split ensures class proportions are preserved in train/test sets.
    X_train, X_test, y_train, y_test = train_test_split(
        x_interval, y, test_size=0.2, random_state=42, stratify=y
    )

    # MLP with one hidden layer of 50 units, trained until convergence (max 1000 iterations).
    model = MLPClassifier(hidden_layer_sizes=(50,), max_iter=1000, random_state=42)
    model.fit(X_train, y_train)

    # Predict on test set and compute error rate (1 - accuracy).
    y_pred = model.predict(X_test)
    return 1 - accuracy_score(y_test, y_pred)

# === Initial evaluations for BO ===
# Evaluate model on all initial sampled centres.
nn_results = [
    {"Interval center": center, "Error Rate": train_nn(x, y, center)}
    for center in sampled_centers
]
nn_df = pd.DataFrame(nn_results).sort_values(by="Interval center")

# GP training data for BO.
x_sample = np.array(nn_df["Interval center"]).reshape(-1, 1)
y_sample = np.array(nn_df["Error Rate"]).reshape(-1, 1)

# === Bayesian Optimisation setup ===
# Use a Matérn kernel (ν=1.5) — common in BO for functions that are not infinitely smooth.
kernel = Matern(nu=1.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, n_restarts_optimizer=10)

# Acquisition function: Expected Improvement (EI).
# This balances exploration (high uncertainty) and exploitation (low predicted error).
def expected_improvement(x_query, x_sample, y_sample, gp, xi=0.01):
    mu, sigma = gp.predict(x_query, return_std=True)
    mu_sample_opt = np.max(y_sample)  # For minimisation, could also use np.min() with sign flip.
    with np.errstate(divide="ignore"):
        z = (mu - mu_sample_opt - xi) / sigma
        ei = (mu - mu_sample_opt - xi) * norm.cdf(z) + sigma * norm.pdf(z)
        ei[sigma == 0.0] = 0.0  # EI is zero when uncertainty is zero.
    return ei

# Select the next candidate point from EI, avoiding already-sampled centres.
def bayesian_sample_one_point(x_sample, y_sample, gp, bounds, tried_points):
    x_candidates = np.arange(int(bounds[0]), int(bounds[1]) + 1).reshape(-1, 1)
    ei_values = expected_improvement(x_candidates, x_sample, y_sample, gp).ravel()

    # Mask out previously sampled points by setting EI to -∞.
    ei_values[[x in tried_points for x in x_candidates.ravel()]] = -np.inf

    if np.all(ei_values == -np.inf):
        return None  # All possible points have been sampled.

    return int(x_candidates[np.argmax(ei_values)][0])

# Variance estimation for stopping rule.
def estimate_variance(error_rates):
    return np.var(error_rates)

# === Main BO loop ===
iteration = 0
while iteration < 100:
    iteration += 1

    # Fit GP surrogate model on observed data.
    gp.fit(x_sample, y_sample)

    # Propose new interval centre using EI.
    bounds = (0, x.shape[1] - 1)
    tried_points = set(nn_df["Interval center"])
    new_center = bayesian_sample_one_point(x_sample, y_sample, gp, bounds, tried_points)
    if new_center is None:
        break

    # Evaluate new point and add to dataset.
    new_error_rate = train_nn(x, y, new_center)
    nn_df = pd.concat(
        [nn_df, pd.DataFrame([{"Interval center": new_center, "Error Rate": new_error_rate}])]
    ).drop_duplicates()

    # Compute GP-based simple regret bound and variance of error rates.
    x_query = np.linspace(bounds[0], bounds[1], 100).reshape(-1, 1)
    mu, sigma = gp.predict(x_query, return_std=True)
    simple_regret_bound = np.min(mu + 1.50 * sigma) - np.min(mu - 1.50 * sigma)
    error_var = estimate_variance(nn_df["Error Rate"])

    print(f"Iteration {iteration}: New Interval Center = {new_center}, Error Rate = {new_error_rate}")
    print(f"Simple Regret Bound: {simple_regret_bound:.4f}, Error Variance: {error_var:.4f}")

    # === Plot GP model fit ===
    plt.figure(figsize=(12, 5))
    plt.plot(x_sample, y_sample, "ro", label="Sampled Points")
    plt.plot(new_center, new_error_rate, "go", label="New Sample")
    plt.plot(x_query, mu, "b-", label="GP Mean Prediction")
    plt.fill_between(x_query.ravel(), mu - 1.96 * sigma, mu + 1.96 * sigma,
                     color="purple", alpha=0.2, label="95% CI")
    plt.title(f"Bayesian Optimisation Iteration {iteration}")
    plt.xlabel("Interval Center")
    plt.ylabel("Error Rate")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    # === Plot EI acquisition function ===
    ei_values = expected_improvement(x_query, x_sample, y_sample, gp)
    plt.figure(figsize=(12, 3))
    plt.plot(x_query, ei_values, "g-", label="Expected Improvement")
    plt.title("Acquisition Function")
    plt.xlabel("Interval Center")
    plt.ylabel("EI")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()

    # Stopping condition based on regret bound vs. variance.
    if simple_regret_bound <= np.sqrt(error_var):
        print("Termination Condition Met: Stopping Bayesian Optimization.")
        break

    # Update BO dataset for next iteration.
    x_sample = np.array(nn_df["Interval center"]).reshape(-1, 1)
    y_sample = np.array(nn_df["Error Rate"]).reshape(-1, 1)

# === Final output ===
best_row = nn_df.loc[nn_df["Error Rate"].idxmin()]
best_center = int(best_row["Interval center"])
best_error = float(best_row["Error Rate"])
best_start, best_end = get_sampled_interval(x, best_center, interval_width)

print(f"\nBest Interval Found:\nCenter: {best_center}  →  Interval: [{best_start}, {best_end})\nError Rate: {best_error:.4f}")


In [ ]:
#CNN AI Data

# ===========================================================
# Optimised CNN + Bayesian Optimisation for Time Series Data
# ===========================================================

# --- Standard library imports ---
import pickle
import numpy as np
import matplotlib.pyplot as plt

# --- Scikit-learn imports for preprocessing, evaluation, and GP regression ---
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern

# --- PyTorch imports for defining and training CNN ---
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


# -----------------------------------------------------------
# Load and preprocess the dataset
# -----------------------------------------------------------
with open("data (1).pkl", "rb") as f:
    data = pickle.load(f)

x = np.array(data['x'])  # Input features (time series)
y = np.array(data['y'])[:x.shape[0]]  # Labels, trimmed to match feature count


# -----------------------------------------------------------
# CNN architecture definition (1D CNN for time series)
# -----------------------------------------------------------
class CNN1D(nn.Module):
    def __init__(self, input_length, laten=10):
        """
        input_length: length of the input time series segment
        laten: number of convolution filters / latent features
        """
        super(CNN1D, self).__init__()
        # First convolution layer: 1 input channel → laten output channels
        self.conv1 = nn.Conv1d(1, laten, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool1d(kernel_size=2)  # Downsampling by factor of 2
        
        # Second convolution layer: laten input channels → laten output channels
        self.conv2 = nn.Conv1d(laten, laten, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool1d(kernel_size=2)  # Downsampling by factor of 2 again
        
        # Flatten layer to feed into fully connected layer
        self.flatten = nn.Flatten()
        
        # Dynamically calculate output size after convolutions/pooling
        conv_output_length = self._get_conv_output_shape(input_length)
        
        # Fully connected layers
        self.fc1 = nn.Linear(conv_output_length, laten)
        self.dropout = nn.Dropout(0.5)  # Regularisation
        self.fc2 = nn.Linear(laten, 1)  # Binary classification output
        self.sigmoid = nn.Sigmoid()  # Output in range [0,1] for probability

    def _get_conv_output_shape(self, input_length):
        """
        Pass a dummy tensor through conv+pool layers to determine
        the flattened output size dynamically.
        """
        dummy_input = torch.zeros(1, 1, input_length)
        x = self.pool1(torch.relu(self.conv1(dummy_input)))
        x = self.pool2(torch.relu(self.conv2(x)))
        return x.view(1, -1).size(1)

    def forward(self, x):
        """
        Forward pass through the CNN.
        """
        x = x.permute(0, 2, 1)  # Rearrange to [batch, channels=1, time]
        x = self.pool1(torch.relu(self.conv1(x)))
        x = self.pool2(torch.relu(self.conv2(x)))
        x = self.flatten(x)
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return self.sigmoid(x)


# -----------------------------------------------------------
# Helper: Get start/end indices for a time series sub-interval
# -----------------------------------------------------------
def get_sampled_interval(data, center, width):
    start = max(0, center - width // 2)
    end = min(data.shape[1], center + width // 2)
    return start, end


# -----------------------------------------------------------
# Train CNN on a specific interval of the time series
# -----------------------------------------------------------
def train_nn(data, labels, center, width, plot=False):
    """
    Trains the CNN on a given segment of the time series defined
    by 'center' and 'width', returns classification error (1-accuracy).
    """
    start, end = get_sampled_interval(data, center, width)
    interval_length = end - start
    
    # If the interval is too short, skip and return max error
    if interval_length < 6:
        return 1.0

    # Extract sub-segment of the time series
    x = data[:, start:end]
    y = labels.astype(np.float32).reshape(-1, 1)

    # Standardise the features
    scaler = StandardScaler().fit(x)
    x = scaler.transform(x)[:, :, np.newaxis]  # Add channel dimension

    # Train/test split (stratified to preserve class balance)
    x_train, x_test, y_train, y_test = train_test_split(
        x, y, stratify=y, test_size=0.2, random_state=42
    )

    # Convert to PyTorch tensors
    x_train = torch.tensor(x_train).float()
    x_test = torch.tensor(x_test).float()
    y_train = torch.tensor(y_train).float()
    y_test = torch.tensor(y_test).float()

    # Instantiate model, loss function, optimiser
    model = CNN1D(input_length=x.shape[1])
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # Wrap in DataLoader for batching
    train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=32, shuffle=True)

    # Train for fixed number of epochs
    model.train()
    for epoch in range(10):
        for xb, yb in train_loader:
            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()

    # Evaluate on test set
    model.eval()
    with torch.no_grad():
        preds = model(x_test).numpy()
        y_pred = (preds > 0.5).astype(int)
    return 1 - accuracy_score(y_test.numpy(), y_pred)  # Return error rate


# -----------------------------------------------------------
# Train CNN on the full time series (baseline model)
# -----------------------------------------------------------
def train_full_nn(data, labels):
    return train_nn(data, labels, center=data.shape[1] // 2, width=data.shape[1])


# -----------------------------------------------------------
# Bayesian Optimisation setup
# -----------------------------------------------------------

interval_width = 10             # Fixed width for sub-intervals
n_initial = 10                  # Number of random starting points
center_bounds = (0, x.shape[1] - 1)  # Bounds for interval center

# Random initial centres
initial_centers = np.random.choice(
    np.arange(center_bounds[0], center_bounds[1]),
    size=n_initial, replace=False
)

# Evaluate CNN error on initial centres
x_sample = initial_centers.reshape(-1, 1)
y_sample = np.array([train_nn(x, y, c, interval_width) for c in initial_centers]).reshape(-1, 1)


# -----------------------------------------------------------
# Expected Improvement (EI) acquisition function
# -----------------------------------------------------------
def expected_improvement(X, X_sample, Y_sample, model, xi=0.01):
    mu, sigma = model.predict(X, return_std=True)
    mu_sample_opt = np.min(Y_sample)  # We're minimising error
    with np.errstate(divide='warn'):
        imp = mu_sample_opt - mu - xi
        Z = imp / sigma
        ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
        ei[sigma == 0.0] = 0.0
    return ei


# -----------------------------------------------------------
# Bayesian Optimisation loop using Gaussian Process surrogate
# -----------------------------------------------------------
kernel = Matern(nu=1.5)  # Matérn kernel (less smooth than RBF)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, n_restarts_optimizer=10)
max_iter = 30  # Number of BO iterations

for _ in range(max_iter):
    gp.fit(x_sample, y_sample)
    
    # Candidate points = all possible centres not yet tried
    grid = np.arange(center_bounds[0], center_bounds[1] + 1).reshape(-1, 1)
    tried = set(x_sample.ravel())
    grid = np.array([pt for pt in grid if pt[0] not in tried])
    if len(grid) == 0:
        break
    
    # Select next point via EI maximisation
    ei = expected_improvement(grid, x_sample, y_sample, gp)
    best_idx = np.argmax(ei)
    next_point = grid[best_idx][0]
    
    # Evaluate CNN on this new centre
    new_error = train_nn(x, y, next_point, interval_width)
    
    # Append to dataset
    x_sample = np.vstack((x_sample, [[next_point]]))
    y_sample = np.append(y_sample, [[new_error]])


# -----------------------------------------------------------
# Report results
# -----------------------------------------------------------
best_idx = np.argmin(y_sample)
best_center = int(x_sample[best_idx][0])
best_start, best_end = get_sampled_interval(x, best_center, interval_width)
best_error = float(y_sample[best_idx])
full_model_error = train_full_nn(x, y)

print("Best interval center:", best_center)
print(f"Best interval range:  ({best_start} to {best_end})")
print(f"Best interval error:  {best_error:.4f}")
print(f"Full model error:     {full_model_error:.4f}")


In [ ]:
#Parametrised CNN ECG Data

# %% ---------------------- imports & seeds ----------------------
# Core Python and numerical libs
import os
import math
import random
import numpy as np

# PyTorch for deep learning
import torch
import torch.nn as nn

# Gaussian Process regression for BO
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel

# Normal distribution functions for Expected Improvement
from scipy.stats import norm

# Load ECG200 dataset: each row is [label, feature1, feature2, ...]
train_data = np.loadtxt("ECG200_TRAIN.txt")
test_data = np.loadtxt("ECG200_TEST.txt")

# Split into features/labels
X_train_full, y_train_full = train_data[:, 1:], train_data[:, 0].astype(int)
X_test_full, y_test_full = test_data[:, 1:], test_data[:, 0].astype(int)
L = X_train_full.shape[1]  # Signal length
SEED = 42


# %% ---------------------- model ----------------------
class TFInspiredCNN(nn.Module):
    """
    CNN inspired by TensorFlow-style tutorials:
    - Two convolution + pooling blocks with padding (so conv layers preserve length when stride=1)
    - Flatten dynamically based on input length
    - Fully connected head for classification
    """
    def __init__(self, in_length, n_classes=2):
        super().__init__()
        # First conv block: 1→16 channels, kernel=5, padding keeps length
        self.conv1 = nn.Conv1d(1, 16, kernel_size=5, padding=2)
        self.pool1 = nn.MaxPool1d(kernel_size=2, stride=2)  # Halves length

        # Second conv block: 16→32 channels
        self.conv2 = nn.Conv1d(16, 32, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool1d(kernel_size=2, stride=2)  # Halves length again

        self.flatten = nn.Flatten()

        # Dynamically determine flattened dimension after conv+pool sequence
        with torch.no_grad():
            _x = torch.zeros(1, 1, in_length)                 # dummy input
            _x = self.pool1(torch.relu(self.conv1(_x)))
            _x = self.pool2(torch.relu(self.conv2(_x)))
            flat_dim = _x.numel()

        # Fully connected layers
        self.fc1 = nn.Linear(flat_dim, 64)
        self.fc2 = nn.Linear(64, n_classes)

    def forward(self, x):
        # Input x: shape [B, L] → add channel dimension → [B, 1, L]
        x = x.unsqueeze(1)
        x = self.pool1(torch.relu(self.conv1(x)))
        x = self.pool2(torch.relu(self.conv2(x)))
        x = self.flatten(x)
        x = torch.relu(self.fc1(x))
        return self.fc2(x)  # logits


# %% ---------------------- windowing utils ----------------------
MIN_WIN = 16  # minimum segment length (ensures valid after pooling)

def clamp_window(center, width, L):
    """
    Given (center, width) and total signal length L:
    - Clamp width to [MIN_WIN, L]
    - Clamp center so window stays inside signal
    - Return start index, end index, and (possibly adjusted) width
    """
    width = int(max(MIN_WIN, min(int(width), L)))
    center = int(np.clip(int(center), 0, L - 1))

    start = center - width // 2
    end = start + width
    # Shift window if it goes out of bounds
    if start < 0:
        start, end = 0, width
    if end > L:
        end, start = L, L - width
    return start, end, width


# %% ---------------------- training helper ----------------------
def train_nn(
    X_train, y_train, X_test, y_test,
    center, width,
    epochs=10, batch_size=32, lr=1e-3, seed=0
):
    """
    Train the CNN on a cropped segment [center, width] of the ECG signal.
    Returns classification error on the (cropped) test set.
    """
    torch.manual_seed(seed)
    L = X_train.shape[1]

    # Crop the specified window
    start, end, width = clamp_window(center, width, L)
    Xtr = X_train[:, start:end]
    Xte = X_test[:, start:end]

    # Instantiate model for this exact input length
    model = TFInspiredCNN(in_length=width, n_classes=int(np.unique(y_train).size))
    model.train()

    # Wrap training data in DataLoader
    Xtr_t = torch.tensor(Xtr, dtype=torch.float32)
    ytr_t = torch.tensor(y_train, dtype=torch.long)
    ds = torch.utils.data.TensorDataset(Xtr_t, ytr_t)
    dl = torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=True)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    # Training loop
    for _ in range(epochs):
        for bx, by in dl:
            optimizer.zero_grad()
            out = model(bx)
            loss = criterion(out, by)
            loss.backward()
            optimizer.step()

    # Evaluate on cropped test set
    model.eval()
    with torch.no_grad():
        Xte_t = torch.tensor(Xte, dtype=torch.float32)
        logits = model(Xte_t)
        preds = logits.argmax(dim=1).cpu().numpy()
    err = float((preds != y_test).mean())
    return err


# %% ---------------------- BO utilities (EI) ----------------------
def expected_improvement(X, model, y_best, xi=0.01):
    """
    Compute Expected Improvement (EI) for minimisation.
    X: candidate points, shape (m, d)
    model: fitted sklearn GP model
    y_best: best observed value (lowest error so far)
    xi: exploration-exploitation trade-off parameter
    """
    mu, sigma = model.predict(X, return_std=True)
    sigma = np.clip(sigma, 1e-12, None)  # avoid division by zero
    imp = y_best - mu - xi                # improvement over best
    Z = imp / sigma
    ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
    return np.maximum(ei, 0.0)

def propose_location(bounds, gp, y_best, n_samp=2000):
    """
    Maximise EI via random search over the given parameter bounds.
    bounds: [(low_center, high_center), (low_width, high_width)]
    gp: fitted GP model
    """
    X0 = np.random.uniform(bounds[0][0], bounds[0][1], size=n_samp)  # candidate centers
    X1 = np.random.uniform(bounds[1][0], bounds[1][1], size=n_samp)  # candidate widths
    Xcand = np.stack([X0, X1], axis=1)
    ei = expected_improvement(Xcand, gp, y_best)
    idx = int(np.argmax(ei))
    return Xcand[idx:idx+1]


# %% ---------------------- Run BO over (center, width) ----------------------
# Bounds for the search space: center ∈ [0, L-1], width ∈ [MIN_WIN, L]
BO_BOUNDS = [(0.0, float(L - 1)), (float(MIN_WIN), float(L))]

# Initial random design
n_init = 8
X_init = np.zeros((n_init, 2), dtype=np.float64)
X_init[:, 0] = np.random.uniform(0, L - 1, size=n_init)        # random centers
X_init[:, 1] = np.random.uniform(MIN_WIN, L, size=n_init)      # random widths

# Evaluate initial design points
Y_init = []
for c, w in X_init:
    err = train_nn(
        X_train_full, y_train_full, X_test_full, y_test_full,
        center=int(c), width=int(w),
        epochs=10, batch_size=32, lr=1e-3, seed=SEED
    )
    Y_init.append(err)

# Initialise sample history
X_sample = X_init.astype(np.float64)
Y_sample = np.array(Y_init, dtype=np.float64)

# GP surrogate: Matérn 5/2 + constant + white noise
kernel = ConstantKernel(1.0, (1e-3, 1e3)) * Matern(nu=2.5) \
         + WhiteKernel(noise_level=1e-6, noise_level_bounds=(1e-9, 1e-3))
gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=3,
                               normalize_y=True, random_state=SEED)

# BO loop
n_iter = 20
for it in range(n_iter):
    gp.fit(X_sample, Y_sample)
    y_best = np.min(Y_sample)
    x_next = propose_location(BO_BOUNDS, gp, y_best, n_samp=4000)

    # Round chosen parameters to integers for training
    c_next, w_next = int(round(x_next[0, 0])), int(round(x_next[0, 1]))

    # Evaluate model at new point
    y_next = train_nn(
        X_train_full, y_train_full, X_test_full, y_test_full,
        center=c_next, width=w_next,
        epochs=10, batch_size=32, lr=1e-3, seed=SEED
    )

    # Append to history
    X_sample = np.vstack([X_sample, x_next])
    Y_sample = np.append(Y_sample, y_next)

    print(f"[{it+1:02d}/{n_iter}] center={c_next:3d}, width={w_next:3d} -> "
          f"test error={y_next:.3f} | best={np.min(Y_sample):.3f}")

# Report best found configuration
best_idx = int(np.argmin(Y_sample))
best_c, best_w = X_sample[best_idx]
best_err = float(np.min(Y_sample))
print("\nBest found:")
print(f"  center={int(round(best_c))}, width={int(round(best_w))}, test error={best_err:.4f}")


In [ ]:
#Parametrised CNN AI Data


# --- Standard library imports ---
import pickle
import numpy as np
import matplotlib.pyplot as plt

# --- Scikit-learn imports for preprocessing, evaluation, and Gaussian Process modelling ---
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
from scipy.stats import norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern

# --- PyTorch imports for defining and training the CNN ---
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


# -----------------------------------------------------------
# Load and preprocess dataset
# -----------------------------------------------------------
with open("data (1).pkl", "rb") as f:
    data = pickle.load(f)

x = np.array(data['x'])                          # Time series data
y = np.array(data['y'])[:x.shape[0]]             # Labels, matched to x length
signal_length = x.shape[1]                       # Total number of time points in each signal


# -----------------------------------------------------------
# Define 1D CNN architecture for binary classification
# -----------------------------------------------------------
class CNN1D(nn.Module):
    def __init__(self, input_length, laten=10):
        """
        input_length: length of the input segment
        laten: number of convolution filters
        """
        super(CNN1D, self).__init__()
        # First convolution layer (1 input channel → laten filters)
        self.conv1 = nn.Conv1d(1, laten, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool1d(kernel_size=2)  # Downsample by 2
        # Second convolution layer (laten → laten filters)
        self.conv2 = nn.Conv1d(laten, laten, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool1d(kernel_size=2)  # Downsample again
        # Flatten for dense layers
        self.flatten = nn.Flatten()
        # Dynamically compute flattened conv output size
        conv_output_length = self._get_conv_output_shape(input_length)
        # Fully connected layers
        self.fc1 = nn.Linear(conv_output_length, laten)
        self.dropout = nn.Dropout(0.5)            # Regularisation
        self.fc2 = nn.Linear(laten, 1)            # Output layer (binary classification)
        self.sigmoid = nn.Sigmoid()               # Output probability

    def _get_conv_output_shape(self, input_length):
        """Pass dummy data through conv/pool stack to get flattened output size."""
        dummy_input = torch.zeros(1, 1, input_length)
        x = self.pool1(torch.relu(self.conv1(dummy_input)))
        x = self.pool2(torch.relu(self.conv2(x)))
        return x.view(1, -1).size(1)

    def forward(self, x):
        """Forward pass."""
        x = x.permute(0, 2, 1)  # [batch, channels=1, time]
        x = self.pool1(torch.relu(self.conv1(x)))
        x = self.pool2(torch.relu(self.conv2(x)))
        x = self.flatten(x)
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        return self.sigmoid(self.fc2(x))


# -----------------------------------------------------------
# Train CNN on a given interval [start, end)
# -----------------------------------------------------------
def train_nn(data, labels, start, end):
    """
    Trains the CNN on a slice of the time series defined by [start, end).
    Returns classification error (1 - accuracy).
    """
    if end - start < 6:  # Too short for two pooling layers
        return 1.0

    # Extract the segment
    x_interval = data[:, start:end]
    y_binary = labels.astype(np.float32).reshape(-1, 1)

    # Standardise features
    scaler = StandardScaler().fit(x_interval)
    x_scaled = scaler.transform(x_interval)[:, :, np.newaxis]

    # Train/test split
    x_train, x_test, y_train, y_test = train_test_split(
        x_scaled, y_binary, stratify=y_binary, test_size=0.2, random_state=42
    )

    # Convert to PyTorch tensors
    x_train = torch.tensor(x_train).float()
    x_test = torch.tensor(x_test).float()
    y_train = torch.tensor(y_train).float()
    y_test = torch.tensor(y_test).float()

    # Create model, loss, optimiser
    model = CNN1D(input_length=x_train.shape[1])
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # DataLoader for batching
    train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=32, shuffle=True)

    # Training loop
    model.train()
    for epoch in range(10):
        for xb, yb in train_loader:
            optimizer.zero_grad()
            preds = model(xb)
            loss = criterion(preds, yb)
            loss.backward()
            optimizer.step()

    # Evaluate
    model.eval()
    with torch.no_grad():
        preds = model(x_test).numpy()
        y_pred = (preds > 0.5).astype(int)
    return 1 - accuracy_score(y_test.numpy(), y_pred)


# -----------------------------------------------------------
# Train CNN on the full signal (baseline)
# -----------------------------------------------------------
def train_full_nn(data, labels):
    return train_nn(data, labels, start=0, end=signal_length)


# -----------------------------------------------------------
# Expected Improvement acquisition function
# -----------------------------------------------------------
def expected_improvement(X, X_sample, Y_sample, model, xi=0.01):
    """
    Compute EI for minimisation.
    X: candidate points
    X_sample: already tried points
    Y_sample: objective values at X_sample
    model: trained GP surrogate
    xi: exploration parameter
    """
    mu, sigma = model.predict(X, return_std=True)
    mu_sample_opt = np.min(Y_sample)  # Best observed value so far
    with np.errstate(divide='warn'):
        imp = mu_sample_opt - mu - xi
        Z = imp / sigma
        ei = imp * norm.cdf(Z) + sigma * norm.pdf(Z)
        ei[sigma == 0.0] = 0.0
    return ei


# -----------------------------------------------------------
# Bayesian Optimisation over all possible (start, end) intervals
# -----------------------------------------------------------
min_width = 6
n_initial = 10
initial_samples = []

# Random initial (start, end) pairs
while len(initial_samples) < n_initial:
    s = np.random.randint(0, signal_length - min_width)
    e = np.random.randint(s + min_width, signal_length + 1)
    initial_samples.append([s, e])

x_sample = np.array(initial_samples)
y_sample = np.array([train_nn(x, y, s, e) for s, e in x_sample]).reshape(-1, 1)

# Gaussian Process surrogate with Matérn kernel
kernel = Matern(nu=1.5)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-6, n_restarts_optimizer=10)
max_iter = 30

# BO loop
for _ in range(max_iter):
    gp.fit(x_sample, y_sample)

    # Generate all possible (start, end) pairs not yet tried
    candidate_grid = np.array([
        [s, e]
        for s in range(0, signal_length - min_width)
        for e in range(s + min_width, signal_length + 1)
        if [s, e] not in x_sample.tolist()
    ])
    if len(candidate_grid) == 0:
        break

    # Evaluate EI for each candidate and pick the best
    ei = expected_improvement(candidate_grid, x_sample, y_sample, gp)
    best_idx = np.argmax(ei)
    next_s, next_e = candidate_grid[best_idx]

    # Train CNN on the chosen interval and update samples
    new_error = train_nn(x, y, next_s, next_e)
    x_sample = np.vstack((x_sample, [[next_s, next_e]]))
    y_sample = np.append(y_sample, [[new_error]])


# -----------------------------------------------------------
# Report the best found interval
# -----------------------------------------------------------
best_idx = np.argmin(y_sample)
best_start, best_end = x_sample[best_idx]
best_error = float(y_sample[best_idx])
full_model_error = train_full_nn(x, y)

print("Best interval start:", best_start)
print("Best interval end:  ", best_end)
print("Best width:         ", best_end - best_start)
print("Best interval error:", round(best_error, 4))
print("Full model error:   ", round(full_model_error, 4))


# -----------------------------------------------------------
# Plot: Average CNN error vs interval width
# -----------------------------------------------------------
widths = x_sample[:, 1] - x_sample[:, 0]                   # Interval widths
unique_widths = np.unique(widths)                          # Unique widths tried
avg_errors = [np.mean(y_sample[widths == w]) for w in unique_widths]

plt.figure(figsize=(12, 6))
plt.bar(unique_widths, avg_errors, color='skyblue', edgecolor='black')
plt.xlabel("Interval Width")
plt.ylabel("Average Classification Error")
plt.title("CNN Error vs Interval Width (Bayesian Optimisation Results)")
plt.grid(True)
plt.tight_layout()
plt.show()
